# BODAQS Data Explorer

This notebook consumes artifacts produced by the batch pre-processing pipeline and displays the data using generic widgets.

 - Metric histogram widget
 - Event browser widget
 - Metric scatter widget
 - Signal histogram widget
 - Session browser widget


### Runtime settings

In [1]:
from pathlib import Path
from IPython.display import display
from bodaqs_analysis.ui import make_preprocess_runtime_settings_editor

runtime_settings_editor = make_preprocess_runtime_settings_editor(
    artifacts_dir=Path("artifacts/Neil"),
    bike_profile_path=None,
    fit_dir=None,
    fit_bindings_path=None,
    prompt_for_descriptions=False,
    show_preprocess_profile_path=False,
    show_generic_log_metadata=False,
    show_bike_profile_path=False,
    show_fit_inputs=False,
    show_prompt_for_descriptions=False,
    show_run_tz_label=False,
)
display(runtime_settings_editor.ui)


### Run selector

In [2]:
from pathlib import Path
from bodaqs_analysis.widgets.session_selector import make_session_selector
from bodaqs_analysis.schema import load_event_schema
from bodaqs_analysis.artifacts import load_session_artifacts
from bodaqs_analysis.widgets.loaders import make_session_loader
    
runtime_settings = runtime_settings_editor.get_settings()
ARTIFACTS_DIR = runtime_settings["artifacts_dir"]

SCHEMA_PATH = Path(r"event schema\event_schema.yaml")
schema = load_event_schema(SCHEMA_PATH)

sel = make_session_selector(artifacts_dir=ARTIFACTS_DIR, select_first_by_default=True)
display(sel["ui"])

events_index_df = sel["get_events_index_df"]()
key_to_ref = sel["get_key_to_ref"]()
store = sel["store"]

session_loader = make_session_loader(store=store, key_to_ref=key_to_ref)


### Signal histogram

In [3]:
from bodaqs_analysis.widgets.signal_histogram_widget import make_signal_histogram_widget_for_loader
from bodaqs_analysis.widgets.signal_histogram_widget import make_signal_histogram_rebuilder

key_to_ref = sel["get_key_to_ref"]()
events_index_df = sel["get_events_index_df"]()
session_loader = make_session_loader(store=store, key_to_ref=key_to_ref)

hist = make_signal_histogram_rebuilder(sel=sel)
display(hist["out"])


Output()

### Event browser

In [4]:
from bodaqs_analysis.widgets.event_browser import make_event_browser_widget_for_loader
from bodaqs_analysis.widgets.event_browser import make_event_browser_rebuilder
from bodaqs_analysis.widgets.loaders import load_all_events_for_selected

key_to_ref = sel["get_key_to_ref"]()
events_index_df = sel["get_events_index_df"]()
session_loader = make_session_loader(store=store, key_to_ref=key_to_ref)

if not key_to_ref:
    raise ValueError("No sessions selected. Select sessions first.")

events_df_sel = load_all_events_for_selected(store, key_to_ref=key_to_ref)

browser = make_event_browser_rebuilder(sel=sel, schema=schema)
display(browser["out"])

Output()

### Metrics scatter plot

In [3]:
from bodaqs_analysis.widgets.metric_scatter_widget import make_metric_scatter_widget_for_loader
from bodaqs_analysis.widgets.metric_scatter_widget import make_metric_scatter_rebuilder

key_to_ref = sel["get_key_to_ref"]()
events_index_df = sel["get_events_index_df"]()
session_loader = make_session_loader(store=store, key_to_ref=key_to_ref)

scatter = make_metric_scatter_rebuilder(sel=sel, schema=schema)
display(scatter["out"])

Output()

### Metrics histogram

In [30]:
from bodaqs_analysis.widgets.metric_histogram_widget import (
    make_metric_histogram_widget_for_loader,
    make_metric_histogram_rebuilder,
)

#key_to_ref = sel["get_key_to_ref"]()
#events_index_df = sel["get_events_index_df"]()
#session_loader = make_session_loader(store=store, key_to_ref=key_to_ref)

mhist = make_metric_histogram_rebuilder(sel=sel, schema=schema)
display(mhist["out"])

Output()

In [14]:
import plotly.io as pio
from bodaqs_analysis.widgets.session_selector import attach_refresh
pio.renderers.default = "notebook_connected"

refresh = attach_refresh(
    sel,
    rebuild_fns=[browser["rebuild"], hist["rebuild"], scatter["rebuild"], mhist["rebuild"]],
)

# optional manual trigger:
# refresh["trigger"]()
# optional detach:
# refresh["detach"]()


### Session window browser

In [5]:
from bodaqs_analysis.widgets.session_window_browser_widget import make_session_window_browser_rebuilder

rb = make_session_window_browser_rebuilder(sel=sel)
display(rb["out"])


Output()

In [ ]:
import ipywidgets as widgets
from IPython.display import display
display(widgets.IntSlider())